In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from scipy.cluster.hierarchy import fcluster

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:
def load_data_all_subjects():
    all_subjects_data = {}
    cfg = load_config()
    for subject_index in cfg.dataset.test_subject_indices:
        predictions, uncertainties, _, ch_names = load_predicted_amplitude_for_subject(subject_index)
    
        

        file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
        epochs = mne.read_epochs(file_path)
        info_subj = epochs.info
        all_subjects_data[subject_index] = {"predictions": predictions, "uncertainties": uncertainties,  "ch_names": ch_names, "info_subj": info_subj}
    return all_subjects_data

In [ ]:
import pickle
import pickle

def load_predicted_amplitude_for_subject(subject_index=2, rep=1):
    data_dir = f"/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/gradshap_explanations_rep_{rep}"
    file_path = os.path.join(data_dir, f"gradshap_data_subject_{subject_index}_rep_{rep}.npy")

    subject_data = np.load(file_path, allow_pickle=True).item()
    predictions, uncertainties, explanations = subject_data['predictions'], subject_data['uncertainties'], subject_data['explanations']

        
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    
    return predictions, uncertainties, explanations, ch_names

In [ ]:
 #   mean_diff_per_channel, median_diff_per_channel = calculate_diff_per_channel(original_predictions, freq_bands, amplification_factors, ch_names, #subject_index=subject_index)
def median_difference_all_subjects_dw_power(data_all_subjects, take_abs=False, rep=1):
    distance_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_power/parallel_perturbation_distance"
    pert_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_power/parallel_perturbation_power"
    median_diff_per_channel_all_subjects = {}
    mean_diff_per_channel_all_subjects = {}
    std_diff_per_channel_all_subjects = {}
    diff_all = {}
    for subject_index, data in data_all_subjects.items():
        diff_all[subject_index] = {}
        predictions = data["predictions"]
        ch_names = data["ch_names"]
        median_diff_per_channel = {}
        mean_diff_per_channel = {}
        std_diff_per_channel = {}
        for band_name, (low_freq, high_freq) in freq_bands.items():
            median_diff_per_channel[band_name] = {}
            mean_diff_per_channel[band_name] = {}
            std_diff_per_channel[band_name] = {}
            diff_all[subject_index][band_name] = {}
            for factor in amplification_factors:
            
                file_path_distance = f"parallel_distance_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}_rep_{rep}.npy"
                load_path_distance = os.path.join(distance_dir, file_path_distance)
                distances = np.load(load_path_distance, allow_pickle=True).item()

                perturbed_data = np.load(f"{pert_dir}/parallel_perturbed_prediction_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}_rep_{rep}.npy", allow_pickle=True).item()

                median_diff_per_channel[band_name][factor] = {}
                mean_diff_per_channel[band_name][factor] = {}
                std_diff_per_channel[band_name][factor] = {}
                diff_all[subject_index][band_name][factor] = {}
                for ch_name in ch_names:
                    diff_all[subject_index][band_name][factor][ch_name] = []
                    perturbed_amplitude = perturbed_data[ch_name]
                    if take_abs:
                        diff = np.abs(predictions - perturbed_amplitude)/distances[ch_name]
                        diff_all[subject_index][band_name][factor][ch_name].append(diff)         
                    else:
                        diff = (predictions - perturbed_amplitude)/distances[ch_name]
                        diff_all[subject_index][band_name][factor][ch_name].append(diff)

                    median_diff_per_channel[band_name][factor][ch_name] = np.median(diff)
                    mean_diff_per_channel[band_name][factor][ch_name] = np.mean(diff)
                    std_diff_per_channel[band_name][factor][ch_name] = np.std(diff)
        median_diff_per_channel_all_subjects[subject_index] = median_diff_per_channel
        mean_diff_per_channel_all_subjects[subject_index] = mean_diff_per_channel
        std_diff_per_channel_all_subjects[subject_index] = std_diff_per_channel
    return median_diff_per_channel_all_subjects, mean_diff_per_channel_all_subjects, std_diff_per_channel_all_subjects, diff_all


In [ ]:
def median_difference_all_subjects_dw_phase(data_all_subjects, amplification_factors,take_abs=False, rep=1):
    distance_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_phase/parallel_perturbation_distance_phase"
    pert_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_phase/parallel_perturbation_phase"
    median_diff_per_channel_all_subjects = {}
    diff_all = {}
    mean_diff_per_channel_all_subjects = {}
    std_diff_per_channel_all_subjects = {}
    for subject_index, data in data_all_subjects.items():
        predictions = data["predictions"]
        ch_names = data["ch_names"]
        median_diff_per_channel = {}
        mean_diff_per_channel = {}
        std_diff_per_channel = {}
        diff_all[subject_index] = {}
        for band_name, (low_freq, high_freq) in freq_bands.items():
            median_diff_per_channel[band_name] = {}
            mean_diff_per_channel[band_name] = {}
            std_diff_per_channel[band_name] = {}
            diff_all[subject_index][band_name] = {}
            for factor in amplification_factors:
            
                file_path_distance = file_path = f"parallel_distance_dict_{band_name}_channel_phase_shift_{factor}°_subject_{subject_index}_rep_{rep}.npy"
                load_path_distance = os.path.join(distance_dir, file_path_distance)
                distances = np.load(load_path_distance, allow_pickle=True).item()

                perturbed_data = np.load(f"{pert_dir}/parallel_perturbed_prediction_dict_{band_name}_channel_phase_shift_{factor}°_subject_{subject_index}_rep_{rep}.npy", allow_pickle=True).item()

                median_diff_per_channel[band_name][factor] = {}
                mean_diff_per_channel[band_name][factor] = {}
                std_diff_per_channel[band_name][factor] = {}
                diff_all[subject_index][band_name][factor] = {}
                for ch_name in ch_names:
                    diff_all[subject_index][band_name][factor][ch_name] = []
                    perturbed_amplitude = perturbed_data[ch_name]
                    if take_abs:
                        diff = np.abs(predictions - perturbed_amplitude)/distances[ch_name]  
                        diff_all[subject_index][band_name][factor][ch_name].append(diff)
                              
                    else:
                        diff = (predictions - perturbed_amplitude)/distances[ch_name]
                        diff_all[subject_index][band_name][factor][ch_name].append(diff)

                    median_diff_per_channel[band_name][factor][ch_name] = np.median(diff)
                    mean_diff_per_channel[band_name][factor][ch_name] = np.mean(diff)
                    std_diff_per_channel[band_name][factor][ch_name] = np.std(diff)
        median_diff_per_channel_all_subjects[subject_index] = median_diff_per_channel
        mean_diff_per_channel_all_subjects[subject_index] = mean_diff_per_channel
        std_diff_per_channel_all_subjects[subject_index] = std_diff_per_channel
    return median_diff_per_channel_all_subjects, mean_diff_per_channel_all_subjects, std_diff_per_channel_all_subjects, diff_all

In [ ]:
freq_bands = {
              "delta": (0, 4),
              "theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}
amplification_factors = [0.5,0.8,0.9,1.1,1.2,1.5]
phase_peturbations = np.arange(45, 316, 45)

In [ ]:
data_all_subjects = load_data_all_subjects()
median_diff_per_channel_all_subjects_dw_power, mean_diff_per_channel_all_subjects_dw_power, std_diff_per_channel_all_subjects_dw_power, diff_all_power = median_difference_all_subjects_dw_power(data_all_subjects, take_abs=True, rep=1)
median_diff_per_channel_all_subjects_dw_phase, mean_diff_per_channel_all_subjects_dw_phase, std_diff_per_channel_all_subjects_dw_phase, diff_all_phase = median_difference_all_subjects_dw_phase(data_all_subjects, phase_peturbations, take_abs=True, rep=1)

In [ ]:
diff_all_power[1]["alpha"][0.8]["Fp1"][0][0],diff_all_power[1]["alpha"][1.2]["Fp1"][0][0],

In [ ]:
diff_all_power[1]["alpha"][0.8]["Fp1"][0][18],diff_all_power[1]["alpha"][1.2]["Fp1"][0][18]

In [ ]:
def plot_power_amplification_effect(subject_id, band, channel, 
                                    diff_data=None, 
                                    amp_factors=None,
                                    figsize=(12, 7),
                                    jitter=0.01,
                                    alpha=0.3,
                                    point_size=20,
                                    show_plot=True):
    """
    Plot the effect of power amplification on a specific channel.
    
    Parameters:
    -----------
    subject_id : int
        The ID of the subject to analyze
    band : str
        The frequency band to analyze (e.g., 'alpha', 'beta', etc.)
    channel : str
        The EEG channel to analyze
    diff_data : dict, optional
        Dictionary containing the difference data. If None, uses diff_all_power
    amp_factors : list or array, optional
        The amplification factors to plot. If None, uses global amplification_factors
    figsize : tuple, optional
        Figure size (width, height) in inches
    jitter : float, optional
        Amount of horizontal jitter to add to points for better visualization
    alpha : float, optional
        Transparency of the scatter points (0-1)
    point_size : int, optional
        Size of the scatter points
    show_plot : bool, optional
        Whether to display the plot using plt.show()
        
    Returns:
    --------
    matplotlib.figure.Figure
        The figure object for further customization if needed
    """
    
    # Use global variables if not specified
    if diff_data is None:
        diff_data = diff_all_power
    
    if amp_factors is None:
        # Use the global amplification_factors
        amp_factors = amplification_factors
        
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # For each amplification factor
    for factor in amp_factors:
        # Get the difference array for this factor
        diff_array = diff_data[subject_id][band][factor][channel][0]
        
        # Create an array of x-values with a small random jitter for better visualization
        x_values = np.full(len(diff_array), factor) + np.random.normal(0, jitter, len(diff_array))
        
        # Plot the points
        ax.scatter(x_values, diff_array, alpha=alpha, s=point_size)
    
    # Add mean values for reference
    means = [np.mean(diff_data[subject_id][band][factor][channel][0]) for factor in amp_factors]
    ax.plot(amp_factors, means, 'r-', linewidth=2, label='Mean')
    
    # Set labels, title, and grid
    ax.set_xlabel('Amplification Factor')
    ax.set_ylabel('Difference Value')
    ax.set_title(f'Effect of Power Amplification on Channel {channel} (Band: {band}, Subject: {subject_id})')
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    # Set x-ticks to amplification factors
    ax.set_xticks(amp_factors)
    
    # Adjust layout
    plt.tight_layout()
    
    # Show the plot if requested
    if show_plot:
        plt.show()
        
    #return fig

# Example usage
plot_power_amplification_effect(1, "alpha", "Fp1")

In [ ]:
# Example usage
plot_power_amplification_effect(1, "alpha", "Fp1",diff_data = diff_all_phase, amp_factors=phase_peturbations, jitter=2)

changing phase whithin a trial has the already shown oscillatory effect (note that while 180 has the strongest effect for many, this likely does not hold in general)

the similarity of results is an effect of taking the mean or median over the trial dimension

In [ ]:
np.mean(diff_all_phase[1]["alpha"][45]["Fp2"][0]),np.mean(diff_all_phase[1]["alpha"][180]["Fp2"][0]),np.mean(diff_all_phase[1]["alpha"][270]["Fp2"][0])

In [ ]:
np.mean(diff_all_power[1]["alpha"][0.5]["Fp2"][0]), np.mean(diff_all_power[1]["alpha"][0.8]["Fp2"][0]),np.mean(diff_all_power[1]["alpha"][1.1]["Fp2"][0])

In [ ]:
np.mean(diff_all_phase[1]["alpha"][45]["Fp1"][0]),np.mean(diff_all_phase[1]["alpha"][180]["Fp1"][0]),np.mean(diff_all_phase[1]["alpha"][270]["Fp1"][0])

In [ ]:
np.mean(diff_all_power[1]["alpha"][0.5]["Fp1"][0]), np.mean(diff_all_power[1]["alpha"][0.8]["Fp1"][0]),np.mean(diff_all_power[1]["alpha"][1.1]["Fp1"][0])